<a href="https://colab.research.google.com/github/jun-yuaka/llm-book/blob/main/LLM_lesson%E2%91%A2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip install transformers[ja,sentencepiece,torch] pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.4/13.4 MB 65.6 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 MB 54.4 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 691.4/691.4 kB 21.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.9/76.9 MB 27.3 MB/s  0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 58.3 MB/s  0:00:00
  Created wheel for ipadic: filename=ipadic-1.0.0-py3-none-any.whl size=13556748 sha256=9b94c128abb8ea4d94184d8f0ee22478afa87782f0e4b72f4b78c373557d5c37
  Stored in directory: /root/.cache/pip/wheels/63/f6/04/86

In [1]:
from transformers import pipeline

In [ ]:
generator = pipeline(
    "text-generation", model="abeja/gpt2-large-japanese"
)

outputs = generator("日本で一番高い山は")
print(outputs[0]["generated_text"])

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: abeja/gpt2-large-japanese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...35}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


日本で一番高い山は富士山 です。 高さは3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山の標高が一番高い山は 富士山 です。 標高は3,524mです。 富士山


### TransformersとGPTの関係、および「デコーダ」について

学習を進める中で、これらの概念のつながりを整理しておくと理解が非常にスムーズになります。

#### 1. Transformers（ライブラリ）とGPTの関係
* **Transformers（Hugging Face社が提供）**: 簡単に言うと、様々なAIモデルを動かすための**「再生プレイヤー（プラットフォーム）」**です。
* **GPT（モデル）**: 再生プレイヤーで流すための**「音楽（具体的なモデルデータ）」**にあたります。

Transformersという共通の仕組み（API）を使うことで、GPTシリーズだけでなく、BERT、Llama、T5など、世界中の研究者が作った多種多様なモデルを、同じような数行のコード（`pipeline`など）で簡単に切り替えて動かすことができます。

----

#### 2. GPTが「デコーダ（Decoder）」であるとはどういうことか？
元々、現在のAI（Transformerアーキテクチャ）には大きく分けて**「エンコーダ」**と**「デコーダ」**という2つの役割があります。

* **エンコーダ（Encoder / 翻訳者・理解者）**: 入力されたテキストの意味を深く「理解・分析」するのが得意なタイプです（例：BERT）。文章の分類や、感情分析、テキストの検索などに使われます。
* **デコーダ（Decoder / 語り手・生成者）**: 与えられた指示や文章の「続き」を1文字（1トークン）ずつ「予測して作り出す（生成する）」のが得意なタイプです（例：GPTシリーズ）。

##### デコーダの仕組み（自動回帰 / Autoregressive）
GPT（デコーダ）は、入力されたテキストに対して**「次に続く確率が最も高い言葉は何か？」**を予測して出力します。

1. 入力：「日本で一番高い山は」
2. GPTの予測 ➡️ 「富士山」を出力
3. 新しい入力：「日本で一番高い山は 富士山」
4. GPTの予測 ➡️ 「です」を出力
5. 新しい入力：「日本で一番高い山は 富士山です」
6. GPTの予測 ➡️ 「。」を出力

このように、自分が直前に出力した言葉を次の入力に巻き込みながら、文章を後ろへ後ろへと紡いでいく（デコードしていく）役割を持っているため、**「デコーダのみ（Decoder-only）のモデル」**と呼ばれています。

In [5]:
import pandas as pd
from transformers import pipeline

# 'cli-tohoku' を正しい組織名 'cl-tohoku' に修正します
fill_mask = pipeline(
    "fill-mask", model="cl-tohoku/bert-base-japanese-v3"
)
masked_text = "日本の首都は[MASK]である"
outputs = fill_mask(masked_text)
display(pd.DataFrame(outputs[:3]))

Loading weights:   0%|          | 0/204 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: cl-tohoku/bert-base-japanese-v3
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,score,token,token_str,sequence
0,0.884170,12569,東京,日本 の 首都 は 東京 で ある
1,0.024820,12759,大阪,日本 の 首都 は 大阪 で ある
2,0.020864,13017,京都,日本 の 首都 は 京都 で ある


In [10]:
masked_text = "今日の映画は刺激的で面白かった。この映画は[MASK]。"
outputs = fill_mask(masked_text)
display(pd.DataFrame(outputs[:3]))

,score,token,token_str,sequence
0,0.683932,23845,素晴らしい,今日 の 映画 は 刺激 的 で 面白かっ た 。 この 映画 は 素晴らしい 。
1,0.101235,24683,面白い,今日 の 映画 は 刺激 的 で 面白かっ た 。 この 映画 は 面白い 。
2,0.048003,26840,楽しい,今日 の 映画 は 刺激 的 で 面白かっ た 。 この 映画 は 楽しい 。
